# installations

In [1]:
# CELL 1: THE FOUNDATION
print("Installing core dependencies...")
!pip install -q silero-vad soundfile torchaudio torch transformers accelerate huggingface_hub
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio
print("Environment ready.")

Installing core dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.7 MB/s eta 0:00:00
Environment ready.


# code

# Hinglish pipeline

In [ ]:
#Handling Hinglish
# EXP: 2 (Native -> English)
# CELL 2: THE MODULAR BACKEND
import io
import torch
import torchaudio
import soundfile as sf
from huggingface_hub import login
from silero_vad import load_silero_vad, get_speech_timestamps
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Configuration ---
HF_TOKEN = "hf_XXXXXXX" # <--- PASTE YOUR TOKEN HERE
login(token=HF_TOKEN)

class AudioProcessor:
    """Handles all raw audio manipulation and physics."""
    def __init__(self):
        self.target_sr = 16000
        self.vad_model = load_silero_vad()

    def process_bytes(self, audio_bytes: bytes) -> torch.Tensor:
        """Converts raw bytes to a clean, trimmed 16kHz tensor."""
        # 1. Read bytes to numpy array
        audio_data, sr = sf.read(io.BytesIO(audio_bytes))
        audio_tensor = torch.tensor(audio_data, dtype=torch.float32)

        # 2. Force Mono
        if audio_tensor.ndim > 1:
            audio_tensor = audio_tensor.mean(dim=1)
        audio_tensor = audio_tensor.unsqueeze(0)

        # 3. Resample if necessary
        if sr != self.target_sr:
            audio_tensor = torchaudio.functional.resample(audio_tensor, orig_freq=sr, new_freq=self.target_sr)

        # 4. Trim Silence (VAD)
        timestamps = get_speech_timestamps(audio_tensor.squeeze(0), self.vad_model)
        if not timestamps:
            return None # No speech found

        speech_chunks = [audio_tensor[0][ts['start']:ts['end']] for ts in timestamps]
        return torch.cat(speech_chunks).unsqueeze(0)


class SpeechEngine:
    """Handles all ML inference (STT and Translation)."""
    def __init__(self):
        print("Booting ML Models into VRAM...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 1. STT Model (AI4Bharat - HINDI)
        self.processor = Wav2Vec2Processor.from_pretrained("ai4bharat/indicwav2vec-hindi", token=HF_TOKEN)
        self.asr_model = Wav2Vec2ForCTC.from_pretrained("ai4bharat/indicwav2vec-hindi", token=HF_TOKEN).to(self.device)

        # 2. UPGRADED Translation Model (Meta NLLB)
        print("Loading NLLB Translator (This is a larger model, takes a moment)...")
        self.trans_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", token=HF_TOKEN)
        self.trans_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", token=HF_TOKEN).to(self.device)

        print("Engines online.")

    def transcribe_and_translate(self, audio_tensor: torch.Tensor) -> dict:
        """Pushes the clean tensor through the ML pipeline and returns both states."""
        if audio_tensor is None:
            return {"hindi": "NO_SPEECH_DETECTED", "english": "NO_SPEECH_DETECTED"}

        audio_tensor = audio_tensor.to(self.device)

        # --- A. Speech to Text (Hindi) ---
        input_values = self.processor(audio_tensor.squeeze(0).cpu().numpy(), return_tensors="pt", sampling_rate=16000).input_values.to(self.device)
        with torch.no_grad():
            logits = self.asr_model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        hindi_text = self.processor.batch_decode(predicted_ids)[0]

        if not hindi_text.strip():
            return {"hindi": "UNINTELLIGIBLE", "english": "UNINTELLIGIBLE"}

        # --- B. Native Translation to English (NLLB) ---
        inputs = self.trans_tokenizer(hindi_text, return_tensors="pt", truncation=True).to(self.device)

        # NLLB requires you to explicitly state the target language ID
        lang_id = self.trans_tokenizer.convert_tokens_to_ids("eng_Latn")

        with torch.no_grad():
            outputs = self.trans_model.generate(**inputs, forced_bos_token_id=lang_id, max_length=128)

        english_text = self.trans_tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Return BOTH strings for debugging and UI validation
        return {"hindi": hindi_text, "english": english_text}


# Initialize global singletons for the API to use
audio_processor = AudioProcessor()
speech_engine = SpeechEngine()

Booting ML Models into VRAM...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading NLLB Translator (This is a larger model, takes a moment)...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Engines online.


# fastapi


In [ ]:
# CELL 3: THE API & TUNNEL
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile, Form
import uvicorn

app = FastAPI(title="Survey PTT Backend")

@app.post("/transcribe_field")
async def process_survey_field(field_id: str = Form(...), audio_file: UploadFile = File(...)):
    """
    Receives a micro-audio clip, cleans it, transcribes it,
    translates it, and returns BOTH Hindi and English text.
    """
    # Read payload
    audio_bytes = await audio_file.read()

    # Execute Pipeline
    clean_tensor = audio_processor.process_bytes(audio_bytes)
    result_dict = speech_engine.transcribe_and_translate(clean_tensor)

    # Return structured JSON with transparency
    return {
        "field_id": field_id,
        "raw_hindi": result_dict["hindi"],
        "transcription_english": result_dict["english"],
        "status": "success" if result_dict["english"] not in ["NO_SPEECH_DETECTED", "UNINTELLIGIBLE"] else "warning"
    }

# Setup ngrok tunnel
NGROK_AUTH_TOKEN = "XXXXXXXXXXXXXXXXXXXXX" # <--- PASTE NGROK TOKEN HERE
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url

print("="*60)
print(f"🚀 API LIVE AT: {public_url}/transcribe_field")
print("="*60)

# Start Server
# Start Server safely inside Colab's existing event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [4878]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 API LIVE AT: https://1d86-34-138-176-128.ngrok-free.app/transcribe_field
INFO:     150.242.151.8:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK
INFO:     150.242.151.8:0 - "POST /transcribe_field HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4878]
